# MLFlow Experiment Logging

In [1]:
import mlflow
from mlflow import MlflowClient
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("mnist-classifier")
client = MlflowClient()
print("Tracking URI:", mlflow.get_tracking_uri())

SEED = 42

Tracking URI: http://localhost:5000


In [2]:
# Loading MNIST data
mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X = mnist.data
y = mnist.target.astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

In [3]:
# Training and evaluation script
def train_and_evaluate(learning_rate, activation, random_state=SEED):
    model = MLPClassifier(
        hidden_layer_sizes=(100,),
        activation=activation,
        learning_rate_init=learning_rate,
        max_iter=50,
        random_state=random_state
    )
    model.fit(X_train, y_train)

    train_preds = model.predict(X_train)
    train_loss = model.loss_
    train_acc = accuracy_score(y_train, train_preds)
    train_f1 = f1_score(y_train, train_preds, average="macro")

    val_proba = model.predict_proba(X_test)
    val_preds = model.predict(X_test)
    val_loss = log_loss(y_test, val_proba)
    val_acc = accuracy_score(y_test, val_preds)
    val_f1 = f1_score(y_test, val_preds, average="macro")

    return model, train_loss, train_acc, train_f1, val_loss, val_acc, val_f1

# Sanity check — no MLflow involved yet
_, train_loss, train_acc, train_f1, val_loss, val_acc, val_f1 = train_and_evaluate(0.1, 'logistic')
print(f"Train loss = {train_loss:.4f}  Train accuracy = {train_acc:.4f}  Train f1_macro = {train_f1:.4f}  ")
print(f"Val loss = {val_loss:.4f}  Val accuracy = {val_acc:.4f}  Val f1_macro = {val_f1:.4f}  ")

Train loss = 1.3122  Train accuracy = 0.5595  Train f1_macro = 0.5372  
Val loss = 1.2189  Val accuracy = 0.5593  Val f1_macro = 0.5386  


In [4]:
# With MLFlow
def train_and_log(learning_rate, activation, run_name=None):
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({"learning_rate": learning_rate, "activation": activation})
        model, train_loss, train_acc, train_f1, val_loss, val_acc, val_f1 = train_and_evaluate(learning_rate=learning_rate, activation=activation)
        mlflow.log_metrics({"Train loss": train_loss, "Train accuracy": train_acc, "Train f1_macro": train_f1})
        mlflow.log_metrics({"Val loss": val_loss, "Val accuracy": val_acc, "Val f1_macro": val_f1})
        mlflow.sklearn.log_model(model, name="model", skops_trusted_types=['sklearn.neural_network._stochastic_optimizers.AdamOptimizer'])

        run_id = mlflow.active_run().info.run_id
        print(f"Logged run id {run_id}")
        print(f"Train loss = {train_loss:.4f}  Train accuracy = {train_acc:.4f}  Train f1_macro = {train_f1:.4f}  ")
        print(f"Val loss = {val_loss:.4f}  Val accuracy = {val_acc:.4f}  Val f1_macro = {val_f1:.4f}  ")

        return run_id

In [5]:
# Sweep over hyperparameters
sweep_run_ids = []
for act in ['relu', 'logistic']:
    for lr in [1e-3, 1e-2, 1e-1]:
        rid = train_and_log(learning_rate=lr, activation=act, run_name=f"mnist-{act}-{lr}")
        sweep_run_ids.append(rid)

print("Sweep run IDs:", sweep_run_ids)

/home/pakshal/lab/assn1/venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


Logged run id 528c02c9375f409fac0f60f1aa9f8817
Train loss = 0.0423  Train accuracy = 0.9875  Train f1_macro = 0.9874  
Val loss = 0.4069  Val accuracy = 0.9595  Val f1_macro = 0.9592  
🏃 View run mnist-relu-0.001 at: http://localhost:5000/#/experiments/1/runs/528c02c9375f409fac0f60f1aa9f8817
🧪 View experiment at: http://localhost:5000/#/experiments/1
Logged run id 1c8289ea1b3c4df4999db6c559f3fe02
Train loss = 0.9250  Train accuracy = 0.6911  Train f1_macro = 0.6842  
Val loss = 1.0025  Val accuracy = 0.6876  Val f1_macro = 0.6799  
🏃 View run mnist-relu-0.01 at: http://localhost:5000/#/experiments/1/runs/1c8289ea1b3c4df4999db6c559f3fe02
🧪 View experiment at: http://localhost:5000/#/experiments/1
Logged run id d98caaa861284387a3ef6a9b0964fb11
Train loss = 2.3120  Train accuracy = 0.1002  Train f1_macro = 0.0183  
Val loss = 2.3076  Val accuracy = 0.0986  Val f1_macro = 0.0179  
🏃 View run mnist-relu-0.1 at: http://localhost:5000/#/experiments/1/runs/d98caaa861284387a3ef6a9b0964fb11
🧪 Vi

/home/pakshal/lab/assn1/venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


Logged run id b6bea55482e04e56a130e5006f2d4878
Train loss = 0.1382  Train accuracy = 0.9619  Train f1_macro = 0.9615  
Val loss = 0.1550  Val accuracy = 0.9542  Val f1_macro = 0.9538  
🏃 View run mnist-logistic-0.001 at: http://localhost:5000/#/experiments/1/runs/b6bea55482e04e56a130e5006f2d4878
🧪 View experiment at: http://localhost:5000/#/experiments/1


/home/pakshal/lab/assn1/venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


Logged run id 80932068fc134b8aa734e6de1f68cece
Train loss = 0.3821  Train accuracy = 0.8879  Train f1_macro = 0.8865  
Val loss = 0.3867  Val accuracy = 0.8849  Val f1_macro = 0.8834  
🏃 View run mnist-logistic-0.01 at: http://localhost:5000/#/experiments/1/runs/80932068fc134b8aa734e6de1f68cece
🧪 View experiment at: http://localhost:5000/#/experiments/1
Logged run id 90ac106dd86f42ddab4c29bd4ba259c1
Train loss = 1.3122  Train accuracy = 0.5595  Train f1_macro = 0.5372  
Val loss = 1.2189  Val accuracy = 0.5593  Val f1_macro = 0.5386  
🏃 View run mnist-logistic-0.1 at: http://localhost:5000/#/experiments/1/runs/90ac106dd86f42ddab4c29bd4ba259c1
🧪 View experiment at: http://localhost:5000/#/experiments/1
Sweep run IDs: ['528c02c9375f409fac0f60f1aa9f8817', '1c8289ea1b3c4df4999db6c559f3fe02', 'd98caaa861284387a3ef6a9b0964fb11', 'b6bea55482e04e56a130e5006f2d4878', '80932068fc134b8aa734e6de1f68cece', '90ac106dd86f42ddab4c29bd4ba259c1']
